# MoodTune — Phase 3: Exploratory Data Analysis

This notebook analyzes the cleaned track-level dataset. Every chart answers a specific question about mood-relevant audio features or recommendation context; no dataset is modified.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

project_roots = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((root for root in project_roots if (root / 'data' / 'processed').is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Run this notebook from the MoodTune project or a child directory.')
df = pd.read_csv(PROJECT_ROOT / 'data' / 'processed' / 'spotify_cleaned.csv')
features = ['valence', 'energy', 'danceability', 'acousticness', 'instrumentalness', 'speechiness', 'loudness', 'tempo', 'popularity', 'liveness']
print(df.shape)
display(df[features].describe().T)

## Distribution question

How are mood-relevant audio features and popularity distributed across the catalogue? Histograms reveal concentration and skew; box plots show the extent of legitimate music-data outliers.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 6))
for axis, column in zip(axes.ravel(), features):
    axis.hist(df[column], bins=40, color='#5B8FF9', edgecolor='white')
    axis.set_title(column.replace('_', ' ').title())
    axis.set_xlabel(column)
    axis.set_ylabel('Tracks')
fig.suptitle('Distribution of Mood-Relevant Features and Popularity', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

box_features = ['tempo', 'loudness', 'duration_ms', 'popularity']
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for axis, column in zip(axes, box_features):
    axis.boxplot(df[column], vert=True)
    axis.set_title(column.replace('_', ' ').title())
    axis.set_ylabel(column)
fig.suptitle('Outlier Context: Values Are Investigated, Not Automatically Removed', y=1.03, fontsize=14)
plt.tight_layout()
plt.show()

## Relationship question

Which feature relationships are strong enough to inform later mood-label methodology, and which should remain secondary signals?

In [ ]:
pairs = [('valence', 'energy'), ('valence', 'danceability'), ('energy', 'loudness'), ('energy', 'acousticness'), ('energy', 'tempo'), ('danceability', 'tempo')]
sample = df.sample(n=min(10000, len(df)), random_state=42)
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for axis, (x_column, y_column) in zip(axes.ravel(), pairs):
    axis.scatter(sample[x_column], sample[y_column], alpha=0.18, s=8, color='#6A5ACD')
    correlation = df[[x_column, y_column]].corr().iloc[0, 1]
    axis.set_title(f'{x_column} vs {y_column} (r={correlation:.3f})')
    axis.set_xlabel(x_column)
    axis.set_ylabel(y_column)
plt.tight_layout()
plt.show()

correlation = df[features].corr()
fig, axis = plt.subplots(figsize=(10, 8))
image = axis.imshow(correlation, cmap='coolwarm', vmin=-1, vmax=1)
axis.set_xticks(range(len(features)), labels=features, rotation=45, ha='right')
axis.set_yticks(range(len(features)), labels=features)
for row in range(len(features)):
    for column in range(len(features)):
        axis.text(column, row, f'{correlation.iloc[row, column]:.2f}', ha='center', va='center', fontsize=8)
fig.colorbar(image, ax=axis, label='Pearson correlation')
axis.set_title('Audio Feature and Popularity Correlations')
plt.tight_layout()
plt.show()

## Genre question

How does track-level de-duplication affect genre coverage, and which genre labels have contrasting average mood-relevant feature profiles? Multi-genre tracks are counted once for each retained label in this descriptive view.

In [ ]:
genre_df = df.assign(track_genre=df['track_genres'].str.split('; ')).explode('track_genre')
genre_counts = genre_df['track_genre'].value_counts()
genre_means = genre_df.groupby('track_genre')[['valence', 'energy', 'danceability', 'popularity']].mean()

fig, axis = plt.subplots(figsize=(10, 6))
genre_counts.sort_values().tail(20).plot.barh(ax=axis, color='#59A14F')
axis.set_title('Twenty Largest Retained Genre Labels by Track Count')
axis.set_xlabel('Track count (multi-genre tracks can appear in multiple labels)')
axis.set_ylabel('Genre')
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
for axis, column in zip(axes.ravel(), genre_means.columns):
    values = genre_means[column].sort_values()
    pd.concat([values.head(10), values.tail(10)]).plot.barh(ax=axis, color='#F28E2B')
    axis.set_title(f'Genre Means: Lowest and Highest {column.title()}')
    axis.set_xlabel(f'Mean {column}')
    axis.set_ylabel('Genre')
plt.tight_layout()
plt.show()